# Regresión Polinomial — Ventas Walmart

**¿Por qué pasar de lineal a polinomial?**  
La Regresión Lineal asume que la relación entre variables es una **línea recta**.  
Cuando los datos tienen curvas, picos o comportamientos estacionales, esa línea no alcanza.

La Regresión Polinomial agrega **columnas X², X³...** que le dan al modelo  
la flexibilidad de ajustarse a esas curvas:
```
Lineal:     ventas = w1·Store + w2·month + b
Polinomial: ventas = w1·Store + w2·Store² + w3·month + w4·month² + b
```

**Dataset:** Walmart.csv — 6,435 registros, 45 tiendas  
**Variable objetivo (Y):** `Weekly_Sales`  
**Herramienta nueva:** `PolynomialFeatures` de sklearn

In [ ]:
# ============================================================
# 1. LIBRERÍAS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures   # ← lo nuevo
from sklearn.pipeline import Pipeline                  # ← encadena pasos
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# ============================================================
# 2. CARGAR Y PREPARAR DATOS
# ============================================================
df = pd.read_csv("Walmart.csv")
df['Date']  = pd.to_datetime(df['Date'], dayfirst=True)
df['month'] = df['Date'].dt.month
df['week']  = df['Date'].dt.isocalendar().week.astype(int)
df = df.drop(columns=['Date'])

print("Columnas:", df.columns.tolist())
print(f"Registros: {len(df)} | Tiendas: {df['Store'].nunique()}")
print(df.head())

In [ ]:
# ============================================================
# 3. ¿QUÉ HACE PolynomialFeatures?
# ============================================================
# Toma tus columnas originales y crea columnas nuevas con potencias.
# Ejemplo con 2 variables (Store, month) y degree=2:
#
#   Original:      [Store, month]
#   Transformado:  [Store, month, Store², Store·month, month²]
#
# Así el modelo puede capturar relaciones curvas sin cambiar el algoritmo.

demo = np.array([[1, 3], [2, 6], [3, 9]])   # 3 filas, 2 columnas
poly_demo = PolynomialFeatures(degree=2, include_bias=False)
demo_transformado = poly_demo.fit_transform(demo)

print("Datos originales (Store, month):")
print(demo)
print("\nDespués de PolynomialFeatures(degree=2):")
print(demo_transformado)
print("\nNombres de las columnas generadas:")
print(poly_demo.get_feature_names_out(['Store', 'month']))
print("\n→ De 2 columnas pasamos a 5. A mayor degree, más columnas se crean.")

In [ ]:
# ============================================================
# 4. GRÁFICA — ventas por semana (muestra la no linealidad)
# ============================================================
# Si los datos tuvieran relación lineal, los puntos formarían
# una línea recta. Si forman una curva → necesitamos polinomial.

ventas_semana = df.groupby('week')['Weekly_Sales'].mean().reset_index()
ventas_mes    = df.groupby('month')['Weekly_Sales'].mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(ventas_semana['week'], ventas_semana['Weekly_Sales'],
                color='steelblue', s=60, edgecolors='white')
axes[0].set_title('Ventas promedio por semana del año', fontsize=13)
axes[0].set_xlabel('Semana (1–52)')
axes[0].set_ylabel('Ventas (USD)')
axes[0].grid(True, alpha=0.3)

meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
axes[1].plot(range(1,13), ventas_mes.values, marker='o',
             color='orangered', linewidth=2)
axes[1].fill_between(range(1,13), ventas_mes.values,
                     ventas_mes.mean(), alpha=0.15, color='orangered')
axes[1].set_xticks(range(1,13))
axes[1].set_xticklabels(meses, rotation=45, fontsize=8)
axes[1].axhline(ventas_mes.mean(), color='gray', linestyle='--',
                linewidth=1, label='Promedio')
axes[1].set_title('Tendencia mensual (no es recta)', fontsize=13)
axes[1].set_ylabel('USD')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Evidencia de no linealidad en los datos', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("→ Los datos no forman una línea recta → la regresión polinomial")
print("  puede ajustarse a estas curvas mejor que la lineal.")

In [ ]:
# ============================================================
# 5. SELECCIONAR X e y — NORMALIZAR — DIVIDIR
# ============================================================
# Mismos pasos que en RLM: los pasos previos al modelo no cambian.

features = ['Store','Holiday_Flag','Temperature',
            'Fuel_Price','CPI','Unemployment','month','week']

X = df[features].values
y = df['Weekly_Sales'].values

X_min, X_max = X.min(axis=0), X.max(axis=0)
y_min, y_max = y.min(), y.max()

X_norm = (X - X_min) / (X_max - X_min)
y_norm = (y - y_min) / (y_max - y_min)

X_train, X_test, y_train, y_test = train_test_split(
    X_norm, y_norm, test_size=0.2, random_state=42
)

print(f"Entrenamiento: {X_train.shape[0]} | Prueba: {X_test.shape[0]}")
print("Normalización y división aplicadas.")

In [ ]:
# ============================================================
# 6. COMPARAR GRADOS — detectar underfitting y overfitting
# ============================================================
# UNDERFITTING → degree muy bajo, el modelo es demasiado simple
#                R² train BAJO + R² test BAJO
#
# BUEN AJUSTE  → R² train y R² test son similares y razonables
#
# OVERFITTING  → degree muy alto, memorizó los datos de entrenamiento
#                R² train MUY ALTO + R² test BAJO
#                (diferencia grande entre ambos = señal de alarma)

print(f"{'Grado':<7} {'R² Train':>10} {'R² Test':>10} {'Diferencia':>12}  Diagnóstico")
print("-" * 60)

mejor_r2 = -999
mejor_grado = 1
resultados = []

for grado in range(1, 5):
    m = Pipeline([
        ('poly', PolynomialFeatures(degree=grado, include_bias=False)),
        ('reg',  LinearRegression())
    ])
    m.fit(X_train, y_train)

    r2_tr = r2_score(y_train, m.predict(X_train))
    r2_te = r2_score(y_test,  m.predict(X_test))
    diff  = r2_tr - r2_te

    diag = '⚠ OVERFITTING'  if diff > 0.1 else \
           '⚠ UNDERFITTING' if r2_te < 0.2 else '✓ Buen ajuste'

    print(f"{grado:<7} {r2_tr:>10.4f} {r2_te:>10.4f} {diff:>12.4f}  {diag}")
    resultados.append((grado, r2_tr, r2_te))

    if r2_te > mejor_r2:
        mejor_r2 = r2_te
        mejor_grado = grado

print(f"\n→ Mejor grado: {mejor_grado} (R² test = {mejor_r2:.4f})")

In [ ]:
# ============================================================
# 7. GRÁFICA — R² train vs test por grado
# ============================================================
# Visualiza dónde empieza el overfitting:
# cuando train sigue subiendo pero test empieza a bajar.

grados    = [r[0] for r in resultados]
r2_trains = [r[1] for r in resultados]
r2_tests  = [r[2] for r in resultados]

plt.figure(figsize=(8, 5))
plt.plot(grados, r2_trains, marker='o', color='steelblue',
         label='R² Entrenamiento', linewidth=2)
plt.plot(grados, r2_tests,  marker='s', color='orangered',
         label='R² Prueba', linewidth=2)
plt.axvline(x=mejor_grado, color='gray', linestyle='--',
            alpha=0.7, label=f'Mejor grado ({mejor_grado})')
plt.fill_between(grados,
                 [max(0, t - 0.005) for t in r2_tests],
                 [min(1, t + 0.005) for t in r2_trains],
                 alpha=0.08, color='gray', label='Zona de diferencia')
plt.title('Underfitting vs Overfitting — R² por grado', fontsize=13)
plt.xlabel('Grado del polinomio')
plt.ylabel('R²')
plt.xticks(grados)
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("→ Líneas juntas = buen ajuste")
print("→ Train sube pero test baja = overfitting")
print("→ Ambas muy bajas = underfitting")

In [ ]:
# ============================================================
# 8. ENTRENAR MODELO FINAL Y MÉTRICAS
# ============================================================
modelo_final = Pipeline([
    ('poly', PolynomialFeatures(degree=mejor_grado, include_bias=False)),
    ('reg',  LinearRegression())
])
modelo_final.fit(X_train, y_train)

predicciones = modelo_final.predict(X_test)
r2      = r2_score(y_test, predicciones)
mae     = mean_absolute_error(y_test, predicciones)
mse     = mean_squared_error(y_test, predicciones)
mae_usd = mae * (y_max - y_min)

print("================================================")
print(f"MÉTRICAS — Polinomial degree={mejor_grado}")
print("================================================")
print(f"R²:        {r2:.4f}  → explica el {r2*100:.1f}% de la variación")
print(f"MAE real:  ${mae_usd:>12,.0f} USD en promedio")
print(f"MSE:       {mse:.4f}")
print()
print("Comparación vs Regresión Lineal Múltiple:")
print(f"  RLM  → R²: 0.1542 | MAE: $433,025")
print(f"  RP   → R²: {r2:.4f} | MAE: ${mae_usd:,.0f}")
mejora = (r2 - 0.1542) / 0.1542 * 100
print(f"\n  → El polinomial mejora el R² en {mejora:.1f}%")

In [ ]:
# ============================================================
# 9. GRÁFICAS DE DIAGNÓSTICO
# ============================================================

y_test_r = y_test        * (y_max - y_min) + y_min
pred_r   = predicciones  * (y_max - y_min) + y_min
residuos = (y_test - predicciones) * (y_max - y_min)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Real vs Predicción
axes[0].scatter(y_test_r, pred_r, alpha=0.3, s=12, color='steelblue')
lim = [y_test_r.min(), y_test_r.max()]
axes[0].plot(lim, lim, 'r--', linewidth=1.5, label='Predicción perfecta')
axes[0].set_title(f'Real vs Predicción — degree={mejor_grado}', fontsize=13)
axes[0].set_xlabel('Ventas reales (USD)')
axes[0].set_ylabel('Ventas predichas (USD)')
axes[0].legend()
axes[0].grid(True, alpha=0.25)

# Residuos
axes[1].scatter(pred_r, residuos, alpha=0.3, s=12, color='orangered')
axes[1].axhline(0, color='black', linewidth=1.2, linestyle='--')
axes[1].set_title('Residuos vs Predicciones', fontsize=13)
axes[1].set_xlabel('Ventas predichas (USD)')
axes[1].set_ylabel('Error (USD)')
axes[1].grid(True, alpha=0.25)

plt.suptitle(f'Diagnóstico — Regresión Polinomial degree={mejor_grado}',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Desviación estándar residuos: ${residuos.std():,.0f} USD")
print("→ Los puntos siguen más cerca de la línea roja vs el modelo lineal.")

In [ ]:
# ============================================================
# 10. PREDICCIÓN — nueva semana
# ============================================================
# El Pipeline aplica PolynomialFeatures automáticamente.
# Solo normalizas el dato nuevo y predices — el resto lo hace el pipeline.

nuevo = {
    'Store': 1, 'Holiday_Flag': 0, 'Temperature': 55.0,
    'Fuel_Price': 3.1, 'CPI': 211.0, 'Unemployment': 8.1,
    'month': 3, 'week': 10
}

X_nuevo      = np.array([[nuevo[f] for f in features]])
X_nuevo_norm = (X_nuevo - X_min) / (X_max - X_min)
pred_norm    = modelo_final.predict(X_nuevo_norm)
pred_real    = (pred_norm[0] * (y_max - y_min)) + y_min

historico = df[df['Store']==nuevo['Store']]['Weekly_Sales'].mean()

print("PREDICCIÓN — Tienda 1, semana 10, marzo")
print(f"  Modelo Polinomial d={mejor_grado}: ${pred_real:>12,.2f} USD")
print(f"  Promedio histórico:       ${historico:>12,.2f} USD")

---
## 📌 Conclusión

| Modelo | R² | MAE real | ¿Cuándo usarlo? |
|---|---|---|---|
| **RLM** | 0.1542 | ~$433,000 | Datos con relación aproximadamente recta |
| **Polinomial d=2** | 0.3064 | ~$389,000 | Datos con curvas, picos o estacionalidad |

**¿Cómo saber si necesitas polinomial?**
- El R² del modelo lineal es bajo (< 0.5 en este tipo de problemas)
- Los residuos tienen un patrón visible (no son aleatorios)
- La gráfica Real vs Predicción tiene puntos muy alejados de la diagonal
- Los datos visualmente forman una curva, no una línea

**Cuidado con el degree:**  
Más alto no siempre es mejor. Comparar R² train vs test en la tabla  
te dice si estás entrando en overfitting.